In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Library

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
tqdm.pandas()


import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input, Flatten, Dense, Embedding, Dropout, Concatenate
from tensorflow.keras.optimizers import Adam

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Utils

In [ ]:
def data_split(df, test_size=0.2, valid_size=0.125, random_state=42):

    # 1차 분할: train+valid, test
    df_train, df_test = train_test_split(df, test_size=test_size, random_state=random_state)

    # 2차 분할: train, valid
    df_train, df_valid = train_test_split(df_train, test_size=valid_size, random_state=random_state)

    # reset index, 혹시나, 없어도 무관
    df_train.reset_index(drop=True, inplace=True)
    df_valid.reset_index(drop=True, inplace=True)
    df_test.reset_index(drop=True, inplace=True)

    'Training Dataset'
    user_id_train = df_train['user_id'].values # 사용자 ID
    item_id_train = df_train['asin'].values # 아이템 ID
    user_bert_train = np.array(df_train['user_bert_emb'].tolist()) # 사용자 BERT
    item_bert_train = np.array(df_train['item_bert_emb'].tolist()) # 사용자 RoBERTa
    user_roberta_train = np.array(df_train['user_roberta_emb'].tolist()) # 아이템 BERT
    item_roberta_train = np.array(df_train['item_roberta_emb'].tolist()) # 아이템 RoBERTa
    y_train = df_train['rating'].values # 평점

    'Valid Dataset'
    user_id_valid = df_valid['user_id'].values # 사용자 ID
    item_id_valid = df_valid['asin'].values # 아이템 ID
    user_bert_valid = np.array(df_valid['user_bert_emb'].tolist()) # 사용자 BERT
    item_bert_valid = np.array(df_valid['item_bert_emb'].tolist()) # 사용자 RoBERTa
    user_roberta_valid = np.array(df_valid['user_roberta_emb'].tolist()) # 아이템 BERT
    item_roberta_valid = np.array(df_valid['item_roberta_emb'].tolist()) # 아이템 RoBERTa
    y_valid = df_valid['rating'].values # 평점

    'Test Dataset'
    user_id_test = df_test['user_id'].values # 사용자 ID
    item_id_test = df_test['asin'].values # 아이템 ID
    user_bert_test = np.array(df_test['user_bert_emb'].tolist()) # 사용자 BERT
    item_bert_test = np.array(df_test['item_bert_emb'].tolist()) # 사용자 RoBERTa
    user_roberta_test = np.array(df_test['user_roberta_emb'].tolist()) # 아이템 BERT
    item_roberta_test = np.array(df_test['item_roberta_emb'].tolist()) # 아이템 RoBERTa
    y_test = df_test['rating'].values # 평점

    return (
        user_id_train, item_id_train, user_bert_train, item_bert_train, user_roberta_train, item_roberta_train, y_train,
        user_id_valid, item_id_valid, user_bert_valid, item_bert_valid, user_roberta_valid, item_roberta_valid, y_valid,
        user_id_test, item_id_test, user_bert_test, item_bert_test, user_roberta_test, item_roberta_test, y_test
    )

# Load Data

In [ ]:
# 데이터 로딩
df = pd.read_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_baseline_emb.pkl')

print(df.shape)
df.head()

In [ ]:
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

df['user_id'] = user_encoder.fit_transform(df['user_id'])
df['asin'] = item_encoder.fit_transform(df['asin'])

# 고유한 user와 item 수를 집계
n_users = df['user_id'].nunique()
print(n_users)
n_items = df['asin'].nunique()
print(n_items)

In [ ]:
user_id_train, item_id_train, user_bert_train, item_bert_train, user_roberta_train, item_roberta_train, y_train, \
user_id_valid, item_id_valid, user_bert_valid, item_bert_valid, user_roberta_valid, item_roberta_valid, y_valid, \
user_id_test, item_id_test, user_bert_test, item_bert_test, user_roberta_test, item_roberta_test, y_test = data_split(df)

# Model

In [ ]:
def MFNR(num_users, num_items, dropout, learning_rate):

    # 사용자 ID
    user_id_input = Input(shape=(1,), name='user_id_input')
    user_emb = Embedding(input_dim=num_users, output_dim=64, name='user_embedding')(user_id_input)
    user_emb = Flatten()(user_emb)

    # 아이템 ID
    item_id_input = Input(shape=(1,), name='item_id_input')
    item_emb = Embedding(input_dim=num_items, output_dim=64, name='item_embedding')(item_id_input)
    item_emb = Flatten()(item_emb)

    # 사용자 멀티 임베딩
    user_bert_input = Input(shape=(768,), name='user_bert_input')
    user_roberta_input = Input(shape=(768,), name='user_roberta_input')

    # 아이템 멀티 임베딩
    item_bert_input = Input(shape=(768,), name='item_bert_input')
    item_roberta_input = Input(shape=(768,), name='item_roberta_input')

    # 사용자 리뷰 MLP
    user_reviews = Concatenate()([user_bert_input, user_roberta_input])

    user_nlp = Dense(512, activation='relu')(user_reviews)
    user_nlp = Dropout(dropout)(user_nlp)
    user_nlp = Dense(256, activation='relu')(user_nlp)
    user_nlp = Dropout(dropout)(user_nlp)
    user_nlp = Dense(128, activation='relu')(user_nlp)
    user_nlp = Dropout(dropout)(user_nlp)

    # 사용자 결합
    user_rep = Concatenate()([user_emb, user_nlp])

    # 아이템 리뷰 MLP
    item_reviews = Concatenate()([item_bert_input, item_roberta_input])

    item_nlp = Dense(512, activation='relu')(item_reviews)
    item_nlp = Dropout(dropout)(item_nlp)
    item_nlp = Dense(256, activation='relu')(item_nlp)
    item_nlp = Dropout(dropout)(item_nlp)
    item_nlp = Dense(128, activation='relu')(item_nlp)
    item_nlp = Dropout(dropout)(item_nlp)

    # 아이템 결합
    item_rep = Concatenate()([item_emb, item_nlp])

    # 예측
    user_item_interaction = Concatenate(name='all_representation')([user_rep, item_rep])
    prediction = Dense(64, activation='relu')(user_item_interaction)
    prediction = Dropout(dropout)(prediction)
    prediction = Dense(32, activation='relu')(prediction)
    prediction = Dropout(dropout)(prediction)
    prediction = Dense(16, activation='relu')(prediction)
    prediction = Dropout(dropout)(prediction)

    final = Dense(1, activation='linear', name='rating_prediction')(prediction)

    model = Model(inputs=[user_id_input, item_id_input, user_bert_input, user_roberta_input, item_bert_input, item_roberta_input],
                  outputs=final)
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mean_squared_error', metrics=['mean_absolute_error', 'mean_squared_error'])

    return model

In [ ]:
dropout = 0.0
learning_rate = 0.0001

In [ ]:
model = MFNR(n_users, n_items, dropout, learning_rate)
model.summary()

In [ ]:
early_stopping = EarlyStopping(monitor ='val_loss', min_delta = 0.001, patience = 5, verbose = 1, mode = 'min', restore_best_weights = True)

In [ ]:
history = model.fit([user_id_train, item_id_train, user_bert_train, item_bert_train, user_roberta_train, item_roberta_train],
                    y_train,
                    validation_data = ([user_id_valid, item_id_valid, user_bert_valid, item_bert_valid, user_roberta_valid, item_roberta_valid], y_valid),
                    epochs = 100,
                    batch_size = 128,
                    callbacks=[early_stopping])

In [ ]:
# 모델 성능 측정
predicted_ratings = model.predict([user_id_test, item_id_test, user_bert_test, item_bert_test, user_roberta_test, item_roberta_test])

MAE = mean_absolute_error(y_test, predicted_ratings)
MSE = mean_squared_error(y_test, predicted_ratings)
RMSE = np.sqrt(MSE)
MAPE = mean_absolute_percentage_error(y_test, predicted_ratings) * 100

print(f'MAE: {MAE:.4f}')
print(f'MSE: {MSE:.4f}')
print(f'RMSE: {RMSE:.4f}')
print(f'MAPE: {MAPE:.4f}%')